In [27]:
!pip -q install openai sentence-transformers

In [28]:
import os, getpass, time
import numpy as np
from openai import OpenAI

def get_key():
    k = os.environ.get("OPENROUTER_API_KEY")
    if k:
        return k
    try:
        from google.colab import userdata
        k = userdata.get("OPENROUTER_API_KEY")
        if k:
            return k
    except Exception:
        pass
    return getpass.getpass("Paste your OpenRouter API key: ")

API_KEY = get_key()
client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=API_KEY)

GEN_MODEL = "openai/gpt-4o-mini"

TOKENS = {"in": 0, "out": 0}

def generate(prompt, system=None, max_new_tokens=300, temperature=0.0):
    msgs = []
    if system:
        msgs.append({"role": "system", "content": system})
    msgs.append({"role": "user", "content": prompt})

    r = client.chat.completions.create(
        model=GEN_MODEL,
        messages=msgs,
        temperature=temperature,
        max_tokens=max_new_tokens,
    )
    if getattr(r, "usage", None):
        TOKENS["in"] += r.usage.prompt_tokens
        TOKENS["out"] += r.usage.completion_tokens
    return r.choices[0].message.content.strip()

print(generate("Reply with exactly: ready", max_new_tokens=6))

ready


In [29]:
DOCS = [
    """CS1001 Introduction to Programming
Week 3: Variables, Types, and Operators

1. Variables
A variable is a name that refers to a value. In Python, assignment uses =
x = 5
y = "hello"

2. Core types
- int: whole numbers, e.g. 42, -7
- float: decimal numbers, e.g. 3.14, 0.5
- str: text, e.g. "CS1001"
- bool: True or False

3. Type conversion
int("5") -> 5
str(5) -> "5"
float("3.14") -> 3.14

4. Operators
Arithmetic: + - * / // % **
Comparison: == != > < >= <=
Logical: and or not

5. Operator precedence
** before * / before + -
Parentheses override precedence.

Worked example:
2 + 3 * 4 = 14
(2 + 3) * 4 = 20

6. Common error
Using = instead of == in a condition.
= assigns, == compares.""",

    """CS1001 Introduction to Programming
Week 4: Control Flow

1. if / elif / else
if score >= 80:
    grade = "A"
elif score >= 70:
    grade = "B"
else:
    grade = "C"

2. while loop
Runs while condition is True.
count = 0
while count < 3:
    print(count)
    count += 1

3. for loop
Iterates over a sequence.
for i in range(3):
    print(i)

4. range()
range(3) -> 0,1,2
range(1,4) -> 1,2,3
range(0,10,2) -> 0,2,4,6,8

5. break and continue
break exits the loop.
continue skips to the next iteration.

6. Common error
Infinite loop: forgetting to update the loop variable.""",

    """CS1001 Introduction to Programming
Week 5: Functions and Scope

1. Defining a function
def add(a, b):
    return a + b

2. Parameters vs arguments
Parameters are in the definition.
Arguments are the values passed in.

3. Return
A function without return returns None.

4. Scope
Variables created inside a function are local.
They are not accessible outside.

5. Default parameters
def greet(name="student"):
    return "Hello, " + name

6. Common error
NameError: using a local variable outside its function.""",

    """MA2001 Linear Algebra for Data Science
Week 2: Vectors and Dot Product

1. Vector
A vector is an ordered list of numbers.
v = [2, 3, 1]

2. Dimension
The number of components. v has dimension 3.

3. Dot product
u dot v = sum of component-wise products.
[1,2] dot [3,4] = 1*3 + 2*4 = 11

4. Norm
||v|| = sqrt(v dot v)
||[3,4]|| = 5

5. Orthogonality
Two vectors are orthogonal if u dot v = 0.

6. Common error
Confusing dot product (scalar) with element-wise product (vector).""",

    """MA2001 Linear Algebra for Data Science
Week 3: Matrices and Matrix Multiplication

1. Matrix
A rectangular array of numbers, size m x n.

2. Matrix multiplication
A (m x n) times B (n x p) gives C (m x p).
C[i][j] = sum over k of A[i][k] * B[k][j]

3. Identity matrix
I has 1s on the diagonal and 0s elsewhere.
A * I = A

4. Transpose
A^T swaps rows and columns.

5. Common error
Multiplying matrices with incompatible dimensions.""",

    """MA2001 Linear Algebra for Data Science
Week 4: Eigenvalues and Eigenvectors

1. Definition
A v = lambda v
v is an eigenvector, lambda is its eigenvalue.

2. Finding eigenvalues
Solve det(A - lambda I) = 0.

3. Example
A = [[2,0],[0,3]]
Eigenvalues: 2 and 3.
Eigenvectors: [1,0] and [0,1].

4. Why it matters
PCA uses eigenvectors of the covariance matrix.

5. Common error
Forgetting that eigenvectors are not unique; any scalar multiple works.""",
]

print(f"{len(DOCS)} documents loaded")

6 documents loaded


In [30]:
CHUNK_WORDS = 60
OVERLAP = 15

def chunk(text, size=CHUNK_WORDS, overlap=OVERLAP):
    words = text.split()
    out, i = [], 0
    while i < len(words):
        out.append(" ".join(words[i:i+size]))
        if i + size >= len(words):
            break
        i += max(1, size - overlap)
    return out

CHUNKS = [(c, d_i) for d_i, d in enumerate(DOCS) for c in chunk(d)]
print(f"{len(DOCS)} docs -> {len(CHUNKS)} chunks")

6 docs -> 13 chunks


In [31]:
from sentence_transformers import SentenceTransformer

EMBEDDER = "sentence-transformers/all-MiniLM-L6-v2"
model = SentenceTransformer(EMBEDDER)

def embed(texts):
    v = model.encode(list(texts), normalize_embeddings=True)
    return np.asarray(v)

MATRIX = embed([c for c, _ in CHUNKS])
print("matrix shape:", MATRIX.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

matrix shape: (13, 384)


In [32]:
TOP_K = 5

def retrieve(question, k=TOP_K):
    q = embed([question])[0]
    scores = MATRIX @ q
    order = np.argsort(-scores)[:k]
    return [(CHUNKS[i][0], CHUNKS[i][1], float(scores[i])) for i in order]

In [33]:
THRESHOLD = 0.20

GROUNDED = (
    "Answer using ONLY the notes provided. "
    "If the notes do not contain the answer, reply exactly: No relevant course content found. "
    "Do not use any other knowledge."
)

def rag_answer(question, k=TOP_K, show=False):
    hits = retrieve(question, k)
    best_score = hits[0][2]

    if best_score < THRESHOLD:
        if show:
            print(f"Q: {question}\nA: No relevant course content found.\n  (top score {best_score:.3f})")
        return "No relevant course content found.", hits

    context = "\n\n".join(f"[{i+1}] {t}" for i, (t, _, _) in enumerate(hits))
    prompt = f"NOTES:\n{context}\n\nQUESTION: {question}"
    out = generate(prompt, system=GROUNDED)

    if show:
        print(f"Q: {question}\nA: {out}\n  (top score {best_score:.3f})")
    return out, hits

In [34]:
rag_answer("What does int('5') return?", show=True)
rag_answer("What is a Python dictionary?", show=True)

Q: What does int('5') return?
A: 5
  (top score 0.484)
Q: What is a Python dictionary?
A: No relevant course content found.
  (top score 0.415)


('No relevant course content found.',
 [('CS1001 Introduction to Programming Week 3: Variables, Types, and Operators 1. Variables A variable is a name that refers to a value. In Python, assignment uses = x = 5 y = "hello" 2. Core types - int: whole numbers, e.g. 42, -7 - float: decimal numbers, e.g. 3.14, 0.5 - str: text, e.g. "CS1001" - bool: True or False',
   0,
   0.41481131315231323),
  ('Variables created inside a function are local. They are not accessible outside. 5. Default parameters def greet(name="student"): return "Hello, " + name 6. Common error NameError: using a local variable outside its function.',
   2,
   0.2740386724472046),
  ('CS1001 Introduction to Programming Week 5: Functions and Scope 1. Defining a function def add(a, b): return a + b 2. Parameters vs arguments Parameters are in the definition. Arguments are the values passed in. 3. Return A function without return returns None. 4. Scope Variables created inside a function are local. They are not accessible o

In [35]:
import pandas as pd

data = [
    # id, course, question, ground_truth, should_abstain
    (1, "CS1001", "What does int(\"5\") return?", "5", False),
    (2, "CS1001", "What is the result of 2 + 3 * 4?", "14", False),
    (3, "CS1001", "Which operator compares equality?", "==", False),
    (4, "CS1001", "What does range(3) produce?", "0,1,2", False),
    (5, "CS1001", "What does break do?", "Exits the loop", False),
    (6, "CS1001", "What does a function return if it has no return statement?", "None", False),
    (7, "CS1001", "Are variables inside a function local or global?", "Local", False),
    (8, "MA2001", "What is [1,2] dot [3,4]?", "11", False),
    (9, "MA2001", "What is the norm of [3,4]?", "5", False),
    (10, "MA2001", "When are two vectors orthogonal?", "When their dot product is 0", False),
    (11, "MA2001", "What size is the product of (m x n) and (n x p)?", "m x p", False),
    (12, "MA2001", "What does A * I equal?", "A", False),
    (13, "MA2001", "What does transpose do?", "Swaps rows and columns", False),
    (14, "MA2001", "What equation finds eigenvalues?", "det(A - lambda I) = 0", False),
    (15, "MA2001", "Which method uses eigenvectors of the covariance matrix?", "PCA", False),
    (16, "CS1001", "Why is = different from ==?", "= assigns, == compares", False),
    (17, "CS1001", "Why does (2+3)*4 differ from 2+3*4?", "Parentheses override precedence", False),
    (18, "CS1001", "Explain what an infinite loop is", "Loop condition never becomes false", False),
    (19, "CS1001", "Explain the difference between break and continue", "break exits loop; continue skips to next iteration", False),
    (20, "CS1001", "Explain scope in one sentence", "Variables inside a function are local", False),
    (21, "CS1001", "Why use default parameters?", "Allow calling without specifying that argument", False),
    (22, "MA2001", "Explain dot product in words", "Sum of component-wise products", False),
    (23, "MA2001", "Explain norm in words", "Square root of the dot product with itself", False),
    (24, "MA2001", "Explain matrix multiplication dimensions", "Inner dimensions must match", False),
    (25, "MA2001", "Explain identity matrix", "1s on diagonal, 0s elsewhere", False),
    (26, "MA2001", "Explain what an eigenvector is", "A vector that only scales under A", False),
    (27, "MA2001", "Explain why eigenvectors are not unique", "Any scalar multiple is also an eigenvector", False),
    (28, "CS1001", "Explain what float(\"3.14\") does", "Converts string to float", False),
    (29, "CS1001", "Explain range(1,4)", "Produces 1,2,3", False),
    (30, "MA2001", "Explain why PCA uses eigenvectors", "They capture directions of maximum variance", False),
    (31, "CS1001", "What does range(0,10,2) produce?", "0,2,4,6,8", False),
    (32, "CS1001", "What does str(5) return?", "5", False),
    (33, "CS1001", "What is 2 ** 3?", "8", False),
    (34, "CS1001", "What does // do?", "Floor division", False),
    (35, "CS1001", "What does % do?", "Modulo", False),
    (36, "MA2001", "What is A^T for [[1,2],[3,4]]?", "[[1,3],[2,4]]", False),
    (37, "MA2001", "Eigenvalues of [[2,0],[0,3]]?", "2 and 3", False),
    (38, "MA2001", "Eigenvectors of [[2,0],[0,3]]?", "[1,0] and [0,1]", False),
    (39, "MA2001", "What is [2,3,1]'s dimension?", "3", False),
    (40, "CS1001", "What does count += 1 do?", "Increments count by 1", False),
    (41, "CS1001", "What is a Python dictionary?", "", True),
    (42, "CS1001", "How do you read a CSV file in pandas?", "", True),
    (43, "CS1001", "What is object-oriented programming?", "", True),
    (44, "CS1001", "How does garbage collection work in Python?", "", True),
    (45, "CS1001", "What is a decorator?", "", True),
    (46, "MA2001", "What is singular value decomposition?", "", True),
    (47, "MA2001", "What is a determinant's geometric meaning?", "", True),
    (48, "MA2001", "How do you compute matrix inverse?", "", True),
    (49, "MA2001", "What is a vector space basis?", "", True),
    (50, "MA2001", "What is the rank of a matrix?", "", True),
    (51, "CS1001", "Tell me about loops", "", True),
    (52, "CS1001", "Explain operators", "", True),
    (53, "MA2001", "Explain vectors", "", True),
    (54, "MA2001", "Tell me about matrices", "", True),
    (55, "CS1001", "What is programming?", "", True),
]

df = pd.DataFrame(data, columns=["id", "course", "question", "ground_truth", "should_abstain"])
df.to_csv("eval_set.csv", index=False)
print("saved", len(df), "questions")

saved 55 questions


In [36]:
def normalize(s):
    return str(s).lower().strip()

answerable_total = 0
correct = 0
abstainable_total = 0
abstained = 0
rows = []

for _, row in df.iterrows():
    q = row["question"]
    expected = normalize(row["ground_truth"])
    should_abstain = bool(row["should_abstain"])

    ans, hits = rag_answer(q)
    ans_norm = normalize(ans)

    if should_abstain:
        abstainable_total += 1
        if "no relevant course content found" in ans_norm:
            abstained += 1
            rows.append((row["id"], "abstained", ""))
        else:
            rows.append((row["id"], "answered", ans[:80]))
    else:
        answerable_total += 1
        if expected and expected in ans_norm:
            correct += 1
            rows.append((row["id"], "correct", ""))
        else:
            rows.append((row["id"], "wrong", ans[:80]))

accuracy = correct / answerable_total if answerable_total else 0
abstention_rate = abstained / abstainable_total if abstainable_total else 0

print(f"Answerable: {answerable_total}, Correct: {correct}, Accuracy: {accuracy:.2%}")
print(f"Abstainable: {abstainable_total}, Abstained: {abstained}, Abstention rate: {abstention_rate:.2%}")

pd.DataFrame(rows, columns=["id", "verdict", "answer_snippet"]).to_csv("eval_results.csv", index=False)
print("saved eval_results.csv")

Answerable: 40, Correct: 19, Accuracy: 47.50%
Abstainable: 15, Abstained: 11, Abstention rate: 73.33%
saved eval_results.csv


In [37]:
def raw_llm(question):
    return generate(question, system="Answer in one short sentence.", max_new_tokens=80)

correct_b = 0
abstained_b = 0

for _, row in df.iterrows():
    q = row["question"]
    expected = normalize(row["ground_truth"])
    should_abstain = bool(row["should_abstain"])

    ans = normalize(raw_llm(q))

    if should_abstain:
        if "no relevant course content found" in ans:
            abstained_b += 1
    else:
        if expected and expected in ans:
            correct_b += 1

print(f"Raw LLM accuracy: {correct_b / answerable_total:.2%}")
print(f"Raw LLM abstention: {abstained_b / abstainable_total:.2%}")

Raw LLM accuracy: 37.50%
Raw LLM abstention: 0.00%


In [38]:
PRICE_IN = 0.15 / 1e6
PRICE_OUT = 0.60 / 1e6

total_cost = TOKENS["in"] * PRICE_IN + TOKENS["out"] * PRICE_OUT
print(f"Tokens in: {TOKENS['in']}, out: {TOKENS['out']}")
print(f"Total cost: ${total_cost:.6f}")
print(f"Cost per query: ${total_cost / 55:.6f}")

Tokens in: 29938, out: 2981
Total cost: $0.006279
Cost per query: $0.000114


In [39]:
print("RAG     | accuracy: {:.2%} | abstention: {:.2%}".format(accuracy, abstention_rate))
print("Raw LLM | accuracy: {:.2%} | abstention: {:.2%}".format(correct_b / answerable_total, abstained_b / abstainable_total))

RAG     | accuracy: 47.50% | abstention: 73.33%
Raw LLM | accuracy: 37.50% | abstention: 0.00%
